In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
%matplotlib inline

from eda import (load_dev, fig_missingness, fig_diurnal,
                 fig_seasonality, fig_stations, fig_persistence)

data = load_dev()

  BEX: 61,368 rows (8,760 sealed in holdout)
  HRL: 61,368 rows (8,760 sealed in holdout)
  KC1: 61,368 rows (8,760 sealed in holdout)
  MY1: 61,368 rows (8,760 sealed in holdout)


# Exploration — development period only

Holdout = 2025, sealed 2026-08-29 (PROJECT_SPEC changelog, "no results seen").
`load_dev()` truncates at 2024-12-31 before anything is plotted, so no figure
here can accidentally read the holdout.

In [2]:
fig_missingness(data)

  wrote results\figures\01_missingness.png
    BEX: worst month 2019-08 at 11%
    HRL: worst month 2019-09 at 71%
    KC1: worst month 2022-01 at 84%
    MY1: worst month 2020-11 at 0%


### What I see

_(write it yourself — outages, which station is worst, anything beyond the known gaps)_

In [3]:
fig_diurnal(data)

    BEX summer NO2 morning peak: 06:00 local
    BEX winter NO2 morning peak: 08:00 local
    HRL summer NO2 morning peak: 07:00 local
    HRL winter NO2 morning peak: 08:00 local
    KC1 summer NO2 morning peak: 07:00 local
    KC1 winter NO2 morning peak: 08:00 local
    MY1 summer NO2 morning peak: 11:00 local
    MY1 winter NO2 morning peak: 09:00 local
  wrote results\figures\02_diurnal.png


### Timezone verdict

Read the **NO₂** peaks, not PM2.5. MY1's kerbside PM2.5 is brake and tyre wear,
not exhaust, so it has almost no diurnal structure — flat there is physics.

PASS = NO₂ morning peak at the same local hour in summer and winter.
FAIL = exactly one hour of offset.

_(write the verdict and the peak hours here, then copy it into
docs/ingest_checks.md §1)_

In [4]:
fig_seasonality(data)
fig_stations(data)

  wrote results\figures\03_seasonality.png
  wrote results\figures\04_stations.png
    station summary (ug/m3):
      BEX: median   6.9  mean   9.6  p95  26.8
      HRL: median   5.7  mean   8.0  p95  22.5
      KC1: median   6.0  mean   8.4  p95  23.5
      MY1: median  10.0  mean  11.9  p95  28.0


### Seasonality and station spread

_(winter above summer? how different are the four stations — does H4 look like a
real transfer test or a trivial one?)_

In [5]:
fig_persistence(data)

    BEX: corr(t, t+6h) = 0.726   persistence MAE = 3.86 ug/m3   (mean level 9.6)
    HRL: corr(t, t+6h) = 0.735   persistence MAE = 3.22 ug/m3   (mean level 8.0)
    KC1: corr(t, t+6h) = 0.732   persistence MAE = 3.37 ug/m3   (mean level 8.4)
    MY1: corr(t, t+6h) = 0.694   persistence MAE = 4.61 ug/m3   (mean level 11.9)
  wrote results\figures\05_persistence.png


### How hard is persistence to beat?

These MAE figures are computed over the whole development period at once.
They are **not** the walk-forward result and must never be quoted as one —
Week 2's harness produces the scoreable number. This is a difficulty gauge.

_(record corr(t, t+6h) and persistence MAE per station — this is what
"F3 works" will have to mean in Week 2)_

In [6]:
import numpy as np
for site, df in data.items():
    m = df.index.month
    for name, months in [("summer", [6,7,8]), ("winter", [12,1,2])]:
        mask = np.isin(m, months)
        curve = df.loc[mask, "no2"].groupby(df.index[mask].hour).mean()
        print(f"{site} {name}: UTC peak {curve.loc[4:12].idxmax():02d}:00, "
              f"local peak {df.loc[mask,'no2'].groupby(df.index[mask].tz_convert('Europe/London').hour).mean().loc[4:12].idxmax():02d}:00")

BEX summer: UTC peak 05:00, local peak 06:00
BEX winter: UTC peak 08:00, local peak 08:00
HRL summer: UTC peak 06:00, local peak 07:00
HRL winter: UTC peak 08:00, local peak 08:00
KC1 summer: UTC peak 06:00, local peak 07:00
KC1 winter: UTC peak 08:00, local peak 08:00
MY1 summer: UTC peak 12:00, local peak 12:00
MY1 winter: UTC peak 09:00, local peak 09:00


In [7]:
Step 1. Write your own commentary in the notebook's markdown cells. In your words, not mine — that's the part you'd be asked to defend. needs to be done by the end of w1 not just for this but for all fo w1 befor emoving onto w2.

SyntaxError: unterminated string literal (detected at line 1) (663078975.py, line 1)